In [ ]:
dev = "xxx"  # key hidden

1. Function to scrape youtube comments from a given Video ID

In [2]:
import googleapiclient.discovery
import pandas as pd

api_service_name = "youtube"
api_version = "v3"
DEVELOPER_KEY = dev

youtube = googleapiclient.discovery.build(
    api_service_name, api_version, developerKey=DEVELOPER_KEY)


def getcomments(video):
  request = youtube.commentThreads().list(
      part="snippet",
      videoId=video,
      maxResults=100
  )

  comments = []

  # Execute the request.
  response = request.execute()

  # Get the comments from the response.
  for item in response['items']:
      comment = item['snippet']['topLevelComment']['snippet']
      public = item['snippet']['isPublic']
      comments.append([
          comment['authorDisplayName'],
          comment['publishedAt'],
          comment['likeCount'],
          comment['textOriginal'],
          comment['videoId'],
          public
      ])

  while (1 == 1):
    try:
     nextPageToken = response['nextPageToken']
    except KeyError:
     break
    nextPageToken = response['nextPageToken']
    # Create a new request object with the next page token.
    nextRequest = youtube.commentThreads().list(part="snippet", videoId=video, maxResults=100, pageToken=nextPageToken)
    # Execute the next request.
    response = nextRequest.execute()
    # Get the comments from the next response.
    for item in response['items']:
      comment = item['snippet']['topLevelComment']['snippet']
      public = item['snippet']['isPublic']
      comments.append([
          comment['authorDisplayName'],
          comment['publishedAt'],
          comment['likeCount'],
          comment['textOriginal'],
          comment['videoId'],
          public
      ])

  df2 = pd.DataFrame(comments, columns=['author', 'updated_at', 'like_count', 'text','video_id','public'])
  return df2

2. Scrape the Video ID's of top n Youtube videos about a given Topic (Most Relevant)

In [5]:
youtube = googleapiclient.discovery.build(
    api_service_name, api_version, developerKey=DEVELOPER_KEY)

def get_top_video_ids(topic, max_results=5):
    request = youtube.search().list(
        part="snippet",
        q=topic,
        type="video",
        maxResults=max_results,
        order="relevance"
    )

    response = request.execute()

    video_ids = []

    for item in response["items"]:
        video_ids.append(item["id"]["videoId"])

    return video_ids

In [11]:
video_ids = get_top_video_ids("interstellar movie explained", 10)

all_dfs = []

for video in video_ids:
    df = getcomments(video)
    all_dfs.append(df)

comments_df = pd.concat(all_dfs, ignore_index=True)

comments_df.head()

,author,updated_at,like_count,text,video_id,public
0,@Looper,2023-05-23T16:18:14Z,687,"Please Note: We meant to say the 21st century,...",QBSw8nSVpmI,True
1,@Tom-zg1vl,2026-05-18T01:09:23Z,0,"Carhartt, “Detroit,” Jacket….",QBSw8nSVpmI,True
2,@randykrus9562,2026-05-14T21:36:41Z,0,"I wanted to love this movie, but in the end......",QBSw8nSVpmI,True
3,@BrianExplores,2026-05-14T11:58:26Z,0,There need to be a part 2,QBSw8nSVpmI,True
4,@cheebaroni5074,2026-05-11T21:06:20Z,0,The only thing worse than food shortages are f...,QBSw8nSVpmI,True


In [12]:
comments_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 23929 entries, 0 to 23928
Data columns (total 6 columns):
 #   Column      Non-Null Count  Dtype
---  ------      --------------  -----
 0   author      23929 non-null  str  
 1   updated_at  23929 non-null  str  
 2   like_count  23929 non-null  int64
 3   text        23929 non-null  str  
 4   video_id    23929 non-null  str  
 5   public      23929 non-null  bool 
dtypes: bool(1), int64(1), str(4)
memory usage: 958.2 KB


Changing the order of the comments according to the Likes

In [14]:
comments_df = comments_df.sort_values(
    by="like_count",
    ascending=False
)

In [18]:
pd.set_option('display.max_colwidth', None)

comments_df.sort_values(
    by="like_count",
    ascending=False
)[["like_count", "text"]].head(20)

,like_count,text
18543,41736,The guy who stay in the ship for 20 years.\n\nHe's the king of quarantine.
15746,17973,My wife keeps asking me to dust my office. I'm like - what if my dad wants to talk to me?
19861,15427,"The scene where cooper comes back to the ship after the ocean planet incident , and sees all the video transmissions from his children gets me every time."
22428,15359,I was disappointed to see that Christopher Nolan did not add a disclaimer to the movie warning people not to go into black holes. Now everyone's going to be doing it.
19994,14772,Did you also find Interstellar confusing?
11014,14772,Did you also find Interstellar confusing?
17695,14460,This movie was made waaaaay too early for its time.
8894,13627,"This shit was filmed in my hometown. If I recall correctly production gave all the fucking crops they actually had to grow for the opening scenes of the film to local farmers to boost the local economy, and I'll always respect how much they helped with that."
2516,10874,You know it's a legendary film when you can make a breakdown of it almost 9 years after its release and people are still thirsty for it.
8940,9933,"""Ya Boi with the single digit IQ"" look at Ya Boi flexing on us."


Convert the Dataframe to CSV file as raw data

In [19]:
comments_df.to_csv(
    "raw_data.csv",
    index=False
)